# Big Data Processing: Pandas and PySpark

**What we'll cover:**
1. Small data → Pandas wins
2. Large data → PySpark required
3. Side-by-side: Same operations in both tools
4. Key concepts: Lazy evaluation, actions vs transformations, partitions
5. Joins: Combining datasets
6. Common beginner mistakes
7. Writing results
8. Your turn: Practice exercises

---

## Setup

In [0]:
import pandas as pd
import time
from pyspark.sql.functions import col, count, avg, sum as spark_sum, desc, when

# Verify Spark is ready (spark is pre-created in Databricks)
print(f"✅ Spark version: {spark.version}")
print(f"✅ Ready to go!")

✅ Spark version: 4.0.0
✅ Ready to go!


### PySpark Data Upload Approach

In [0]:
path = "dbfs:/databricks-datasets/wine-quality/winequality-red.csv"


start = time.time()

df_spark = spark.read.csv(path, header=True, inferSchema=True, sep=";")

In [0]:
display(df_spark)

fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8,5
7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.997,3.26,0.65,9.8,5
11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8,6
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
7.4,0.66,0.0,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
7.9,0.6,0.06,1.6,0.069,15.0,59.0,0.9964,3.3,0.46,9.4,5
7.3,0.65,0.0,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
7.5,0.5,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.8,10.5,5


### PySpark Approach

In [0]:
result = df_spark.filter(col('chlorides') > 0.1) \
    .groupBy('quality') \
    .count() \
    .orderBy(desc('count')) \
    .limit(10)

result.show()

pyspark_time = time.time() - start

print(f"\n  PySpark: {pyspark_time:.2f} seconds")

+-------+-----+
|quality|count|
+-------+-----+
|      5|  116|
|      6|   75|
|      7|   20|
|      4|    8|
|      3|    4|
+-------+-----+


  PySpark: 2.76 seconds


---
## Part 2: Large Data (1 GB+) - PySpark Required

**Dataset:** 10x duplicated flights  
**Size:** ~1 GB, 14M rows  
**Task:** Same analysis on bigger data

**What happens:** Pandas will be slow/crash. PySpark handles it easily.

In [0]:
# Create large dataset by duplicating
print("Creating large dataset (10x duplication)...")

df_base = spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv", 
                          header=True, inferSchema=True, sep = ";")

range_df = spark.range(0, 3000)
df_large = df_base.crossJoin(range_df).drop("id")
# df_large = df_base
# for i in range(3000):
#     df_large = df_large.union(df_base)

print("Created ~1 GB dataset with 14M rows")

Creating large dataset (10x duplication)...
Created ~1 GB dataset with 14M rows


In [0]:
df_large.display

<bound method apply_dataframe_display_patch.<locals>.df_display of DataFrame[fixed acidity: double, volatile acidity: double, citric acid: double, residual sugar: double, chlorides: double, free sulfur dioxide: double, total sulfur dioxide: double, density: double, pH: double, sulphates: double, alcohol: double, quality: int]>

In [0]:
# PYSPARK: Process large data
start = time.time()

#df_large = df_large.select(*[col(c).alias(c.strip()) for c in df_large.columns])

result_large = df_large.filter(col("chlorides") > 0.1) \
    .groupby(col("quality")) \
    .count() \
    .orderBy(desc('count')) \
    .limit(10)


large_time = time.time() - start

print(f"\n⏱️  PySpark on 1 GB: {large_time:.2f} seconds")
print(f"\n💡 Pandas would:")
print(f"   - Take 3-5x longer ({large_time*4:.1f}s estimate)")
print(f"   - Or crash with MemoryError")
print(f"\n   PySpark handles it easily! ✅")


⏱️  PySpark on 1 GB: 0.00 seconds

💡 Pandas would:
   - Take 3-5x longer (0.0s estimate)
   - Or crash with MemoryError

   PySpark handles it easily! ✅


---
## Part 3: Side-by-Side Comparison

Let's see common operations in both tools.

### Operation 1: Select Columns

In [0]:
# PYSPARK  
df_spark_select = df_spark.select('chlorides', 'pH', 'sulphates', 'quality')
print("PySpark:")
df_spark_select.show(5)

PySpark:
+---------+----+---------+-------+
|chlorides|  pH|sulphates|quality|
+---------+----+---------+-------+
|    0.076|3.51|     0.56|      5|
|    0.098| 3.2|     0.68|      5|
|    0.092|3.26|     0.65|      5|
|    0.075|3.16|     0.58|      6|
|    0.076|3.51|     0.56|      5|
+---------+----+---------+-------+
only showing top 5 rows


### Operation 2: Filter Rows

In [0]:
# PYSPARK
df_spark_filter = df_spark.filter(
    (col('chlorides') > 0.1) & 
    (col('sulphates') > 0.1)
)
print(f"PySpark: {df_spark_filter.count():,} chlorides with >0.1 ppm and sulphates with >0.1 ppm")

PySpark: 223 chlorides with >0.1 ppm and sulphates with >0.1 ppm


### Operation 3: Add New Column

In [0]:
# PYSPARK: Returns new dataframe (immutable)
df_spark_with_hours = df_spark.withColumn('Cl_ppb', col('chlorides') / 1000)
print("PySpark:")
df_spark_with_hours.select('chlorides', 'Cl_ppb').show(5)

PySpark:
+---------+--------------------+
|chlorides|              Cl_ppb|
+---------+--------------------+
|    0.076|              7.6E-5|
|    0.098|9.800000000000001E-5|
|    0.092|              9.2E-5|
|    0.075|              7.5E-5|
|    0.076|              7.6E-5|
+---------+--------------------+
only showing top 5 rows


### Operation 4: Group By & Aggregate

In [0]:
# PYSPARK
spark_agg = df_spark.groupBy('quality').agg(
    avg('chlorides').alias('chlorides_avg'),
    count('*').alias('count')
)
print("PySpark:")
spark_agg.show(5)

PySpark:
+-------+-------------------+-----+
|quality|      chlorides_avg|count|
+-------+-------------------+-----+
|      4|0.09067924528301884|   53|
|      8|0.06844444444444445|   18|
|      6|0.08495611285266458|  638|
|      3|0.12250000000000001|   10|
|      7|0.07658793969849244|  199|
+-------+-------------------+-----+
only showing top 5 rows


### Operation 5: Sort

In [0]:
# PYSPARK
spark_sorted = df_spark.orderBy(desc('quality')).limit(5)
print("PySpark - Top 5 quality wines:")
spark_sorted.select('chlorides', 'sulphates', 'quality').show()

PySpark - Top 5 quality wines:
+---------+---------+-------+
|chlorides|sulphates|quality|
+---------+---------+-------+
|    0.072|     0.82|      8|
|    0.073|     0.82|      8|
|    0.045|     0.82|      8|
|    0.078|     0.86|      8|
|    0.086|     0.69|      8|
+---------+---------+-------+



### Summary: Syntax Comparison

| Operation | Pandas | PySpark |
|-----------|--------|----------|
| **Select** | `df[['col1', 'col2']]` | `df.select('col1', 'col2')` |
| **Filter** | `df[df['col'] > 10]` | `df.filter(col('col') > 10)` |
| **Add column** | `df['new'] = df['old'] * 2` | `df.withColumn('new', col('old') * 2)` |
| **Group by** | `df.groupby('col').sum()` | `df.groupBy('col').sum()` |
| **Sort** | `df.sort_values('col')` | `df.orderBy('col')` |

💡 **Key difference:** PySpark uses `col()` function for columns

---
## Part 4: Key PySpark Concepts

Understanding these concepts is critical for working effectively with PySpark.

### Concept 1: Lazy Evaluation

**Pandas:** Executes immediately  
**PySpark:** Builds a plan, executes when needed

In [0]:

# Lazy Evaluation Example 

# Intentionally inefficient order:
# 1. Create column for ALL rows (expensive)
# 2. Filter afterwards (wasteful)

df = spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv", 
                          header=True, inferSchema=True, sep = ";")

query = (df
    .select('quality', 'chlorides', 'sulphates', 'pH')
    .withColumn('quality_wine',
                when(col('chlorides') > 0.1, 'good')
                .when(col('chlorides') > 0.05, 'average')
                .otherwise('bad'))
    .filter(col('chlorides') > 0.1)
    .filter(col('sulphates') > 0.1)         # Written AFTER withColumn   
    .groupBy('quality_wine')
    .agg(avg('chlorides').alias('Cl_avg'), count('*').alias('count'))
    .orderBy(desc('Cl_avg'))
)

display(query)

quality_wine,Cl_avg,count
good,0.1592690582959641,223


### Concept 2: Actions vs Transformations (CRITICAL!)

This is the most important concept in PySpark!

**Transformations** (lazy - just build the plan):
- `.filter()`, `.select()`, `.groupBy()`, `.join()`, `.withColumn()`
- Don't execute anything
- Return a new DataFrame

**Actions** (trigger execution):
- `.count()`, `.show()`, `.collect()`, `.write()`
- Actually run the computation
- Return results to driver

In [0]:
# Actions vs Transformations

import time

df_base = spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv", 
                          header=True, inferSchema=True, sep = ";")

# Transformations - lazy (just build a plan)
start = time.time()
filtered = df.filter(col('quality') > 5)
selected = filtered.select('chlorides', 'quality', 'sulphates')
print(f"Transformations: {time.time() - start:.4f}s")

# Actions - eager (trigger execution)
print("\nAction 1:")
start = time.time()
count = selected.count()
print(f"count() = {count} rows, took {time.time() - start:.2f}s")

print("\nAction 2:")
start = time.time()
selected.show(5)
print(f"show() took {time.time() - start:.2f}s")

print("\nNotice: Each action re-executes the transformations!")

Transformations: 0.0002s

Action 1:
count() = 855 rows, took 0.35s

Action 2:
+---------+-------+---------+
|chlorides|quality|sulphates|
+---------+-------+---------+
|    0.075|      6|     0.58|
|    0.065|      7|     0.47|
|    0.073|      7|     0.57|
|    0.092|      7|     0.75|
|    0.341|      6|     1.08|
+---------+-------+---------+
only showing top 5 rows
show() took 0.25s

Notice: Each action re-executes the transformations!


### Quick Reference: Actions vs Transformations

| Type | Operations | What They Do |
|------|-----------|-------------|
| **Transformations** | `.filter()`, `.select()`, `.groupBy()`, `.join()`, `.withColumn()`, `.orderBy()` | Build execution plan (lazy) |
| **Actions** | `.count()`, `.show()`, `.collect()`, `.take()`, `.first()`, `.write()` | Trigger execution |

**Rule of thumb:** If it returns a DataFrame, it's a transformation. If it returns a value or writes data, it's an action.

### Concept 3: Partitions - How Spark Parallelizes

Data is split into **partitions** - think of them as chunks that can be processed in parallel.

- More partitions = more parallelism = faster (usually)
- But too many = overhead from coordination
- Rule of thumb: 2-4 partitions per CPU core

In [0]:
import sys

print(sys.getrecursionlimit())

3000


In [0]:
print("📦 Partitions Demo\n")
print("💡 Partitions split your data into chunks for parallel processing")
print("   - Databricks serverless automatically manages partitions")
print("   - You can still manually repartition if needed\n")

# Repartition to different sizes
print("Testing different partition strategies...\n")

# Fewer partitions (4)
df_few = df_large.repartition(2)
start = time.time()
df_few.filter(col('quality') > 5).count()
few_time = time.time() - start
print(f"✓ Fewer partitions (4): {few_time:.2f}s")

📦 Partitions Demo

💡 Partitions split your data into chunks for parallel processing
   - Databricks serverless automatically manages partitions
   - You can still manually repartition if needed

Testing different partition strategies...

✓ Fewer partitions (4): 0.99s


In [0]:

# More partitions (4)
df_many = df_large.repartition(4, 'quality')
start = time.time()
df_many.filter(col('quality') > 5).count()
many_time = time.time() - start
print(f"✓ More partitions (4): {many_time:.2f}s")

# Default (let Spark decide)
start = time.time()
df_large.filter(col('quality') > 5).count()
default_time = time.time() - start
print(f"✓ Default (auto): {default_time:.2f}s")

print(f"\n💡 Key takeaways:")
print(f"   - More partitions ≠ always faster")
print(f"   - Too few = not enough parallelism")
print(f"   - Too many = coordination overhead")
print(f"   - Usually best to let Spark decide!")
print(f"\n📝 Note: On standard clusters, you can check partition count with:")
print(f"   df.rdd.getNumPartitions() (not available on serverless)")

✓ More partitions (4): 0.47s
✓ Default (auto): 0.35s

💡 Key takeaways:
   - More partitions ≠ always faster
   - Too few = not enough parallelism
   - Too many = coordination overhead
   - Usually best to let Spark decide!

📝 Note: On standard clusters, you can check partition count with:
   df.rdd.getNumPartitions() (not available on serverless)


### Concept 4: Filter Early = Faster

Always filter data BEFORE expensive operations like groupBy or join.

In [0]:
print("🎯 Optimization: Filter Early\n")

# BAD: Group all 14M rows first
start = time.time()
bad = df_large.groupBy('quality').count().filter(col('count') > 100).count()
bad_time = time.time() - start
print(f"Bad:  {bad_time:.2f}s (grouped all data)")

# GOOD: Filter to 7M rows first
start = time.time()
good = df_large.filter(col('chlorides') > 0.1).groupBy('quality').count().filter(col('count') > 100).count()
good_time = time.time() - start
print(f"Good: {good_time:.2f}s (filtered first)")

print(f"\n⚡ {bad_time/good_time:.1f}x speedup by filtering early!")

🎯 Optimization: Filter Early

Bad:  0.54s (grouped all data)
Good: 0.42s (filtered first)

⚡ 1.3x speedup by filtering early!


---
## Part 5: Joins - Combining DataFrames

Joins are one of the most common operations in real-world data work.

### Create Sample Airport Information Dataset

In [0]:
# Create a small airport info dataset
element_data = [
    (5, 9.4, 0.1),
    (5, 9.8, 0.01),
    (5, 9.8, 0.3),
    (6, 9.8, 0.01),
    (5, 9.4, 0.2),
    (5, 9.4, 0.05),
    (5, 9.4, 0.09)
]


element_df = spark.createDataFrame(element_data, ['quality', 'alcohol', 'radium'])
print("Element Information:")
element_df.show()

Element Information:
+-------+-------+------+
|quality|alcohol|radium|
+-------+-------+------+
|      5|    9.4|   0.1|
|      5|    9.8|  0.01|
|      5|    9.8|   0.3|
|      6|    9.8|  0.01|
|      5|    9.4|   0.2|
|      5|    9.4|  0.05|
|      5|    9.4|  0.09|
+-------+-------+------+



### Left Join - Keep All Flights, Add Info When Available

In [0]:
# Assuming you want the 'quality' and 'alcohol' from the main DataFrame (df_large)
# and 'radium' from the joined DataFrame (element_df).

result_left = df_large.join(
    element_df,
    (df_large.quality == element_df.quality) & (df_large.alcohol == element_df.alcohol),
    'left'
).select(
    df_large.quality,  # Use df_large.quality to avoid ambiguity
    df_large.alcohol, # Use df_large.alcohol to avoid ambiguity
    element_df.radium  # Use element_df.radium
)

print("All wines (with radium info when available):")
result_left.show(5)

All wines (with radium info when available):
+-------+-------+------+
|quality|alcohol|radium|
+-------+-------+------+
|      5|    9.4|  0.09|
|      5|    9.8|   0.3|
|      5|    9.8|   0.3|
|      6|    9.8|  0.01|
|      5|    9.4|  0.09|
+-------+-------+------+
only showing top 5 rows


### Join Types Quick Reference

| Join Type | Keeps | Use When |
|-----------|-------|----------|
| **inner** | Only matches | Need complete info for both sides |
| **left** | All from left + matches from right | Keep all primary data, enrich with lookup |
| **right** | All from right + matches from left | Rarely used (just flip to left) |
| **outer** | Everything from both | Need to see all data, matched or not |

## Part 7: Running SQL Queries in PySpark

PySpark allows you to use SQL syntax alongside DataFrame operations. This is useful if you're more comfortable with SQL or need to work with existing SQL queries.

**Key steps:**
1. **Register a DataFrame as a temporary view** using `.createOrReplaceTempView()`
2. **Run SQL queries** using `spark.sql()`
3. **Mix and match** - You can combine SQL queries with DataFrame operations

The result of `spark.sql()` is a DataFrame, so you can chain additional transformations or actions on it.

In [0]:
# Running SQL queries in PySpark

df = spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv", 
                          header=True, inferSchema=True, sep = ";")

# Register DataFrame as a temporary view
df.createOrReplaceTempView("wines")

# Now you can use SQL
result_1 = spark.sql("""
    SELECT quality, 
           AVG(chlorides) as avg_cl,
           COUNT(*) as wine_count
    FROM wines
    WHERE chlorides > 0.1
    GROUP BY quality
    ORDER BY avg_cl DESC
    LIMIT 10
""")

result_1.show()

# Can also mix SQL and DataFrame operations
#spark.sql("SELECT * FROM flights WHERE delay > 200").filter(col('distance') > 1000).show(5)

+-------+-------------------+----------+
|quality|             avg_cl|wine_count|
+-------+-------------------+----------+
|      3|            0.18725|         4|
|      4|0.18450000000000003|         8|
|      5|0.16190517241379307|       116|
|      6| 0.1588133333333334|        75|
|      7|0.13000000000000003|        20|
+-------+-------------------+----------+



In [0]:
# Register DataFrame as a temporary view
df.createOrReplaceTempView("wine_df")

# Now you can use SQL
result_2 = spark.sql("""
    SELECT pH, 
           AVG(sulphates) as avg_sulphates
    FROM wine_df
    WHERE sulphates > 0.1
    GROUP BY pH
    ORDER BY avg_sulphates DESC
    LIMIT 10
""")

result_2.show()

+----+------------------+
|  pH|     avg_sulphates|
+----+------------------+
|2.74|               2.0|
|2.93|              1.96|
|2.87|              1.36|
| 2.9|              1.33|
| 3.0|1.0616666666666668|
|2.99|             1.025|
|2.94|0.9800000000000001|
|3.04|0.9430000000000002|
|3.03|0.9133333333333334|
|3.11|0.9088888888888891|
+----+------------------+



---
## Part 8: Writing Results

You've processed your data - now save it!

### Write to Table

In [0]:
print("💾 Writing Results\n")

# --- First Query Result ---
# result_1 is the DataFrame from: 
# SELECT quality, AVG(chlorides) as avg_cl, COUNT(*) as wine_count 
# FROM wines WHERE chlorides > 0.1 GROUP BY quality ORDER BY avg_cl DESC LIMIT 10

# SIMPLEST SOLUTION: Save as a table (no file path issues)
table_name = "wine_chlorides_analysis"

result_1.write.mode("overwrite").saveAsTable(table_name)

print(f"✅ Successfully saved as table: {table_name}")
print(f"   You can now query it with: spark.table('{table_name}')")
print(f"   Or use SQL: SELECT * FROM {table_name}")

# Display the results
print("\n📊 Table Contents:")
display(spark.table(table_name))

# Optional: If you need CSV, you can download from the display above
# Click the download icon in the table view

💾 Writing Results

✅ Successfully saved as table: wine_chlorides_analysis
   You can now query it with: spark.table('wine_chlorides_analysis')
   Or use SQL: SELECT * FROM wine_chlorides_analysis

📊 Table Contents:


quality,avg_cl,wine_count
3,0.18725,4
4,0.18450000000000003,8
5,0.16190517241379307,116
6,0.1588133333333334,75
7,0.13000000000000003,20


In [0]:
df_exp = df_large.filter(col('chlorides') > 0.1) \
    .groupBy('quality') \
    .count() \
    .orderBy(desc('count')) \
    .limit(10)

df_exp.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonTopK(sortOrder=[count#288362L DESC NULLS LAST], partitionOrderCount=0)
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#163902]
               +- PhotonShuffleExchangeSink SinglePartition
                  +- PhotonTopK(sortOrder=[count#288362L DESC NULLS LAST], partitionOrderCount=0)
                     +- PhotonProject [quality#286831, count(1)#288363L AS count#288362L]
                        +- PhotonGroupingAgg(keys=[quality#286831], functions=[finalmerge_count(merge count#288365L) AS count(1)#288363L])
                           +- PhotonShuffleExchangeSource
                              +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#163892]
                                 +- PhotonShuffleExchangeSink hashpartitioning(quality#286831, 1024)
                                    +- PhotonGrou